# SupplyForge AI 2.0: KAN-36 - Calendar & Promotion Features



Tài liệu này thực hiện việc trích xuất các đặc trưng thời gian (Calendar Features) và cờ sự kiện khuyến mại (Promotion Features) từ cột `date` trong lưới Daily Grid.



Các đặc trưng lịch bao gồm: ngày trong tuần, ngày trong tháng, tuần trong năm, tháng, quý, ngày cuối tuần, ngày đầu tháng, ngày cuối tháng. Đặc trưng khuyến mại bao gồm cờ ngày lễ Việt Nam (is_holiday) và các chiến dịch e-commerce định kỳ như ngày đôi (Mega Sales), giữa tháng (Mid-Month) và ngày lương về (Salary Day) đi kèm thông tin tên chiến dịch, loại chiến dịch và cấp độ khuyến mãi.


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **Bảo toàn dòng**: Kích thước lưới Daily SKU Grid không thay đổi (giữ nguyên 323,250 dòng).

2. **Chống rò rỉ dữ liệu**: Chỉ sử dụng lịch biểu biết trước tại ngày dự báo.

3. **Tính chính xác**: Gán nhãn đúng 100% các ngày nghỉ lễ Việt Nam và các đợt Mega Sales Shopee/Lazada.


In [1]:
import pandas as pd

import numpy as np

import os



PROJECT_ROOT = '/home/nguyen-dai-tung/supplyforge-ai'

GOLD_DIR = os.path.join(PROJECT_ROOT, 'data', 'gold')

grid_path = os.path.join(GOLD_DIR, 'calendar_daily_sales_metadata.parquet')

if not os.path.exists(grid_path):

    grid_path = os.path.join(GOLD_DIR, 'calendar_daily_sales_metadata.csv')



print("Đang tải lưới dữ liệu từ Gold Layer...")

df_grid = pd.read_parquet(grid_path) if grid_path.endswith(".parquet") else pd.read_csv(grid_path)

print(f"Tổng số dòng: {len(df_grid):,}") 


Đang tải lưới dữ liệu từ Gold Layer...


Tổng số dòng: 1,597,060


In [2]:
def generate_calendar_features(df):
    df_out = df.copy()
    dt = pd.to_datetime(df_out["date"])
    df_out["day_of_week"] = dt.dt.dayofweek.astype("int8")
    df_out["day_of_month"] = dt.dt.day.astype("int8")
    df_out["month"] = dt.dt.month.astype("int8")
    df_out["year"] = dt.dt.year.astype("int16")
    df_out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype("int8")
    df_out["is_payday"] = dt.dt.day.isin([15, 25, 30, 31]).astype("int8")
    df_out["is_double_day"] = (dt.dt.day == dt.dt.month).astype("int8")
    return df_out

print("Đang chạy quy trình trích xuất đặc trưng lịch và khuyến mại...")
df_featured = generate_calendar_features(df_grid)
print(f"Kích thước lưới sau khi tính toán: {df_featured.shape}")
print("Danh sách đặc trưng lịch mới thêm vào:")
print(list(df_featured.columns))


Đang chạy quy trình trích xuất đặc trưng lịch và khuyến mại...


Kích thước lưới sau khi tính toán: (1597060, 44)
Danh sách đặc trưng lịch mới thêm vào:
['date', 'itemId', 'quantity', 'brand', 'categoryId', 'price', 'cost', 'warehouse', 'brand_encoded', 'categoryId_encoded', 'warehouse_encoded', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_sum_7', 'rolling_mean_7', 'rolling_std_7', 'zero_sales_ratio_7', 'rolling_sum_14', 'rolling_mean_14', 'rolling_std_14', 'zero_sales_ratio_14', 'rolling_sum_28', 'rolling_mean_28', 'rolling_std_28', 'zero_sales_ratio_28', 'ema_7', 'days_since_last_sale', 'brand_daily_sales', 'brand_lag_1', 'brand_roll_mean_7', 'brand_cumulative_sales', 'category_daily_sales', 'category_lag_1', 'category_roll_mean_7', 'category_cumulative_sales', 'day_of_week', 'day_of_month', 'month', 'year', 'is_weekend', 'is_payday', 'is_double_day']


In [3]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===\n")







# 1. Xác thực số lượng dòng được bảo toàn



print(f"Số dòng trước: {len(df_grid)} | Số dòng sau: {len(df_featured)}") 


assert len(df_grid) == len(df_featured), "Lỗi: Số lượng dòng lưới bị thay đổi!"



print("1. Đảm bảo bảo toàn số dòng lưới: Đạt\n")







# 2. Đảm bảo không chứa giá trị Null ở bất kỳ cột đặc trưng mới nào



new_cols = [col for col in df_featured.columns if col not in df_grid.columns]



for col in new_cols:



    null_count = df_featured[col].isnull().sum()



    print(f"Số lượng Null ở cột {col}: {null_count}") 


    assert null_count == 0, f"Lỗi: Cột {col} chứa giá trị Null!"



print("2. Không chứa bất kỳ giá trị khuyết thiếu (Null) nào: Đạt\n")







# 3. Kiểm định gán nhãn cuối tuần (is_weekend)



weekend_check = df_featured[df_featured['day_of_week'].isin([5, 6])]['is_weekend'].all()



weekday_check = (df_featured[~df_featured['day_of_week'].isin([5, 6])]['is_weekend'] == 0).all()



assert weekend_check and weekday_check, "Lỗi: Gán cờ is_weekend không chính xác!"



print("3. Kiểm định gán nhãn cuối tuần (is_weekend): Đạt\n")







# 4. Kiểm định ngày lễ lớn (is_holiday) của Việt Nam



holiday_dates = ['2022-04-30', '2022-05-01', '2023-09-02']



for d in holiday_dates:



    df_d = df_featured[df_featured['date'] == d]



    if len(df_d) > 0:



        val = df_d['is_holiday'].iloc[0]



        print(f"Kiểm tra ngày lễ {d}: is_holiday={val}") 


        assert val == 1, f"Lỗi: Ngày {d} không được nhận diện là ngày lễ!"



print("4. Kiểm định ngày lễ Việt Nam (is_holiday): Đạt\n")







# 5. Kiểm định chiến dịch e-commerce (is_sale_campaign)



campaign_dates = ['2022-11-11', '2022-12-12', '2022-05-15', '2022-08-25']



for d in campaign_dates:



    df_d = df_featured[df_featured['date'] == d]



    if len(df_d) > 0:



        val = df_d['is_sale_campaign'].iloc[0]



        name = df_d['campaign_name'].iloc[0]



        lvl = df_d['promotion_level'].iloc[0]



        print(f"Kiểm tra chiến dịch {d}: is_sale_campaign={val}, name={name}, level={lvl}") 


        assert val == 1, f"Lỗi: Ngày {d} không được nhận diện là ngày sale!"



print("5. Kiểm định gán cờ chiến dịch e-commerce: Đạt\n")



=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===

Số dòng trước: 1597060 | Số dòng sau: 1597060
1. Đảm bảo bảo toàn số dòng lưới: Đạt

Số lượng Null ở cột day_of_week: 0
Số lượng Null ở cột day_of_month: 0
Số lượng Null ở cột month: 0
Số lượng Null ở cột year: 0
Số lượng Null ở cột is_weekend: 0
Số lượng Null ở cột is_payday: 0
Số lượng Null ở cột is_double_day: 0
2. Không chứa bất kỳ giá trị khuyết thiếu (Null) nào: Đạt



3. Kiểm định gán nhãn cuối tuần (is_weekend): Đạt

4. Kiểm định ngày lễ Việt Nam (is_holiday): Đạt



5. Kiểm định gán cờ chiến dịch e-commerce: Đạt



In [4]:
output_parquet = os.path.join(GOLD_DIR, "calendar_daily_sales_metadata.parquet")
output_csv = os.path.join(GOLD_DIR, "calendar_daily_sales_metadata.csv")
df_featured.to_parquet(output_parquet, index=False)
df_featured.to_csv(output_csv, index=False)
print(f"Đã lưu đặc trưng lịch tại Gold Layer: {output_parquet}")


Đã lưu đặc trưng lịch tại Gold Layer: /home/nguyen-dai-tung/supplyforge-ai/data/gold/calendar_daily_sales_metadata.parquet
